In [5]:
import os                                                                                                                                                                                                          
from dotenv import load_dotenv
from pathlib import Path
# from polygon import RESTClient
import requests

load_dotenv(Path(".env"))

True

In [2]:
# client = RESTClient(os.getenv("POLYGON_API_KEY"))

In [3]:

# Define the URL and parameters
url = "https://api.polygon.io/fed/v1/inflation-expectations"
params = {
    "date.gte": "2024-12-01",
    "limit": 100,
    "sort": "date.asc",
    "apiKey": os.getenv("POLYGON_API_KEY")
}

# Make the GET request
response = requests.get(url, params=params)

# Check for a successful response
if response.status_code == 200:
    data = response.json()
    # Print or process the inflation expectations
    print(data)
else:
    print(f"Request failed with status code {response.status_code}: {response.text}")


{'status': 'OK', 'request_id': 'd8ee6ec87e5b4a42ac44d7f9486a1d1e', 'results': [{'date': '2024-12-01', 'market_5_year': 2.36, 'market_10_year': 2.3, 'forward_years_5_to_10': 2.24, 'model_1_year': 2.6503563, 'model_5_year': 2.3497474, 'model_10_year': 2.3244686, 'model_30_year': 2.4423094}, {'date': '2025-01-01', 'market_5_year': 2.49, 'market_10_year': 2.4, 'forward_years_5_to_10': 2.31, 'model_1_year': 2.6269584, 'model_5_year': 2.459387, 'model_10_year': 2.4442606, 'model_30_year': 2.5234}, {'date': '2025-02-01', 'market_5_year': 2.61, 'market_10_year': 2.42, 'forward_years_5_to_10': 2.24, 'model_1_year': 2.7276623, 'model_5_year': 2.4909487, 'model_10_year': 2.467667, 'model_30_year': 2.5360973}, {'date': '2025-03-01', 'market_5_year': 2.53, 'market_10_year': 2.33, 'forward_years_5_to_10': 2.14, 'model_1_year': 2.1838527, 'model_5_year': 2.2687087, 'model_10_year': 2.295656, 'model_30_year': 2.440264}, {'date': '2025-04-01', 'market_5_year': 2.35, 'market_10_year': 2.24, 'forward_yea

In [1]:
import pandas as pd
import requests
from io import BytesIO
import warnings

# Suppress SSL warning
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# Download the Excel file (ignore SSL certificate validation)
url = "https://www.bcra.gob.ar/Pdfs/PublicacionesEstadisticas/historico-relevamiento-expectativas-mercado.xlsx"
response = requests.get(url, verify=False)

if response.status_code == 200:
    excel_file = BytesIO(response.content)
    
    # Load the specific sheet, using the second row as header (row 2 in Excel is index 1)
    df = pd.read_excel(excel_file, sheet_name="Base de Datos Completa", header=1)
    
    # Optional cleanup: drop fully empty rows and unnamed columns
    df = df.dropna(how='all')  # Drop fully empty rows
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]  # Drop unnamed columns

else:
    print(f"Failed to download Excel file. Status code: {response.status_code}")


/home/chalito/.pyenv/versions/dbt-analytics/lib/python3.12/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [2]:
df.shape

(6781, 14)

In [5]:
df.columns

Index(['Fecha de pronóstico', 'Variable', 'Referencia', 'Período', 'Mediana',
       'Promedio', 'Desvío', 'Máximo', 'Mínimo', 'Percentil 90',
       'Percentil 75', 'Percentil 25', 'Percentil 10',
       'Cantidad de participantes'],
      dtype='object')

In [6]:
df.sample()

,Fecha de pronóstico,Variable,Referencia,Período,Mediana,Promedio,Desvío,Máximo,Mínimo,Percentil 90,Percentil 75,Percentil 25,Percentil 10,Cantidad de participantes
5706,2024-04-30,Precios minoristas (IPC núcleo; INDEC),var. % i.a.; abr-26,Próx. 24 meses,38.0,41.5,19.6,100.0,7.9,59.4,50.5,29.7,23.4,23


In [10]:
name_map = {
    'Precios minoristas (IPC nivel general-GBA; INDEC)': 'ipc_general_gba',
    'Precios minoristas (IPC nivel general; INDEC)': 'ipc_general',
    'Precios minoristas (IPC núcleo-GBA; INDEC)': 'ipc_nucleo_gba',
    'Precios minoristas (IPC núcleo; INDEC)': 'ipc_nucleo',
    'Tasa de política monetaria (Lebac)': 'tasa_monetaria_lebac',
    'Tasa de política monetaria (Pase 7 días)': 'tasa_monetaria_pase7',
    'Tasa de política monetaria (LELIQ)': 'tasa_monetaria_leliq',
    'Tasa de interés (LELIQ)': 'tasa_interes_leliq',
    'Tasa de interés (BADLAR)': 'tasa_interes_badlar',
    'Tasa de interés (TAMAR)': 'tasa_interes_tamar',
    'Tipo de cambio nominal': 'tipo_cambio',
    'Resultado primario del SPNF': 'resultado_primario',
    'Resultado Primario del SPNF': 'resultado_primario',
    'PIB a precios constantes': 'pib_constantes',
    'Exportaciones': 'exportaciones',
    'Importaciones': 'importaciones',
    'Desocupación abierta': 'desocupacion',
    'desocupación abierta': 'desocupacion',
}


df['Variable'] = df['Variable'].map(name_map)

In [11]:
df

,Fecha de pronóstico,Variable,Referencia,Período,Mediana,Promedio,Desvío,Máximo,Mínimo,Percentil 90,Percentil 75,Percentil 25,Percentil 10,Cantidad de participantes
0,2016-06-30,ipc_general_gba,var. % mensual,2016-07-01 00:00:00,2.20,2.200000,0.400000,3.600000,1.600000,2.700000,2.500000,1.900000,1.800000,39
1,2016-06-30,ipc_general_gba,var. % mensual,2016-08-01 00:00:00,1.90,1.900000,0.300000,3.300000,1.100000,2.200000,2.000000,1.700000,1.600000,39
2,2016-06-30,ipc_general_gba,var. % mensual,2016-09-01 00:00:00,1.70,1.800000,0.300000,3.000000,1.300000,2.000000,1.900000,1.600000,1.500000,39
3,2016-06-30,ipc_general_gba,var. % mensual,2016-10-01 00:00:00,1.60,1.600000,0.300000,2.500000,1.000000,2.000000,1.800000,1.500000,1.300000,39
4,2016-06-30,ipc_general_gba,var. % mensual,2016-11-01 00:00:00,1.50,1.500000,0.200000,2.000000,0.900000,1.900000,1.700000,1.400000,1.300000,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6776,2025-06-30,pib_constantes,var. % trim. s.e.,Trim. III-25,0.74,0.862134,0.622765,3.105819,0.200000,1.628989,0.975000,0.500000,0.275611,35
6777,2025-06-30,pib_constantes,var. % trim. s.e.,Trim. IV-25,0.60,0.595673,0.508174,1.580000,-0.500000,1.099600,0.900000,0.382370,-0.090000,35
6778,2025-06-30,pib_constantes,var. % prom. anual,2025,5.00,4.914643,0.434046,5.939973,4.000000,5.393950,5.200000,4.587500,4.443748,40
6779,2025-06-30,pib_constantes,var. % prom. anual,2026,3.50,3.481701,0.963474,5.928118,1.000000,4.456225,4.000000,3.100000,2.300000,37
